<a href="https://colab.research.google.com/github/Markkwell/Project-YpYt/blob/main/project1_by_Marko_Deko.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Your path Your trials





In [2]:
"""
Your Path Your Trials
Система анализа Steam-отзывов
"""

import re

import numpy as np
import pandas as pd


DATASET_PATH = "steam_reviews.csv"

REQUIRED_COLUMNS = (
    "app_id",
    "app_name",
    "review_text",
    "review_score",
    "review_votes"
)


def load_dataset(path: str) -> pd.DataFrame:
    """
    Загружает CSV-файл и проверяет наличие обязательных столбцов.

    Parameters
    ----------
    path : str
        Путь к файлу с датасетом.

    Returns
    -------
    pd.DataFrame
        Загруженный DataFrame.

    Raises
    ------
    ValueError
        Если файл отсутствует, пустой или имеет неверную структуру.
    """

    try:
        df = pd.read_csv(path)

    except FileNotFoundError:
        raise ValueError(f"Файл не найден: {path}")

    if df.empty:
        raise ValueError("Датасет пуст.")

    missing_columns = [
        column for column in REQUIRED_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Отсутствуют обязательные столбцы: "
            f"{missing_columns}"
        )

    return df


def clean_text(text: str) -> str:
    """
    Подготавливает текст отзыва к анализу.

    Функция переводит текст в нижний регистр,
    удаляет лишние символы и нормализует пробелы.

    Parameters
    ----------
    text : str
        Исходный текст отзыва.

    Returns
    -------
    str
        Очищенный текст.
    """

    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z ]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def prepare_reviews(df: pd.DataFrame) -> pd.DataFrame:
    """
    Выполняет предварительную обработку отзывов.

    Создает очищенный текст и определяет тип отзыва
    (Positive / Negative).

    Parameters
    ----------
    df : pd.DataFrame
        Исходный датасет.

    Returns
    -------
    pd.DataFrame
        Подготовленный DataFrame.
    """

    prepared_df = df.copy()

    prepared_df["app_name"] = prepared_df["app_name"].fillna("Unknown")
    prepared_df["review_text"] = prepared_df["review_text"].fillna("")

    prepared_df["clean_review"] = [
        clean_text(text)
        for text in prepared_df["review_text"]
    ]

    prepared_df["sentiment"] = np.where(
        prepared_df["review_score"] > 0,
        "Positive",
        "Negative"
    )

    return prepared_df


def get_game_rating(df: pd.DataFrame) -> pd.DataFrame:
    """
    Считает статистику отзывов для каждой игры.

    Для каждой игры определяется:
    - число Positive отзывов;
    - число Negative отзывов;
    - отношение Positive/Negative;
    - общее количество отзывов.

    Parameters
    ----------
    df : pd.DataFrame
        Подготовленный DataFrame.

    Returns
    -------
    pd.DataFrame
        Таблица со статистикой игр.
    """

    grouped = (
        df.groupby(["app_name", "sentiment"])
        .size()
        .unstack(fill_value=0)
    )

    grouped["Positive"] = grouped.get("Positive", 0)
    grouped["Negative"] = grouped.get("Negative", 0)

    grouped["ratio"] = grouped["Positive"] / (
        grouped["Negative"] + 1
    )

    grouped["total_reviews"] = (
        grouped["Positive"] + grouped["Negative"]
    )

    return grouped.reset_index()


def show_reviews(df: pd.DataFrame, sentiment: str) -> None:
    """
    Показывает несколько отзывов выбранного типа.

    Parameters
    ----------
    df : pd.DataFrame
        Подготовленный DataFrame.
    sentiment : str
        Positive или Negative.
    """

    reviews = df[
        df["sentiment"] == sentiment
    ][["app_name", "review_text"]].head(5)

    print(f"\n{sentiment} reviews:\n")

    for _, row in reviews.iterrows():
        print(f"Game: {row['app_name']}")
        print(f"Review: {row['review_text']}")
        print("-" * 40)


def top_r_games(df: pd.DataFrame) -> None:
    """
    Показывает 5 игр с лучшим отношением
    положительных и отрицательных отзывов.
    """

    stats = get_game_rating(df)
    result = stats.sort_values(
        by="ratio",
        ascending=False
    ).head(5)

    print("\nTop R:\n")

    for _, row in result.iterrows():
        print(
            f"{row['app_name']} | "
            f"Positive={row['Positive']} | "
            f"Negative={row['Negative']} | "
            f"P/N={row['ratio']:.2f}"
        )


def lead_comm(df: pd.DataFrame) -> None:
    """
    Показывает 5 игр с наибольшим количеством отзывов.
    """

    stats = get_game_rating(df)
    result = stats.sort_values(
        by="total_reviews",
        ascending=False
    ).head(5)

    print("\nLead comm:\n")

    for _, row in result.iterrows():
        print(
            f"{row['app_name']} | "
            f"Reviews={row['total_reviews']}"
        )


def random_comm(
    df: pd.DataFrame,
    game_name: str,
    review_count: int
) -> None:
    """
    Показывает случайные отзывы выбранной игры.

    Parameters
    ----------
    game_name : str
        Название игры.
    review_count : int
        Количество случайных отзывов.
    """

    result = df[
        df["app_name"].str.contains(
            game_name,
            case=False,
            na=False
        )
    ]

    if result.empty:
        print("Игра не найдена.")
        return

    sample_size = min(review_count, len(result))

    sample = result.sample(
        sample_size,
        random_state=np.random.randint(0, 1000)
    )

    print("\nRandom comments:\n")

    for _, row in sample.iterrows():
        print(f"[{row['sentiment']}] {row['review_text']}")
        print("-" * 40)


def game_list(df: pd.DataFrame) -> None:
    """
    Показывает список игр без повторений.
    """

    games = sorted(df["app_name"].dropna().unique())

    print("\nGame list:\n")

    for game in games:
        print(game)


def compare_games(
    df: pd.DataFrame,
    game_1: str,
    game_2: str
) -> None:
    """
    Сравнивает две игры по числу Positive и Negative отзывов.

    Parameters
    ----------
    game_1 : str
        Название первой игры.
    game_2 : str
        Название второй игры.
    """

    stats = get_game_rating(df)

    games = stats[
        stats["app_name"].isin(
            [game_1, game_2]
        )
    ]

    if len(games) < 2:
        print("Одна из игр не найдена.")
        return

    print("\nCompare:\n")

    for _, row in games.iterrows():
        print(
            f"{row['app_name']} | "
            f"Positive={row['Positive']} | "
            f"Negative={row['Negative']} | "
            f"P/N={row['ratio']:.2f}"
        )


def print_menu() -> None:
    """
    Показывает список доступных команд.
    """

    print("\nКоманды:")
    print("Positive reviews")
    print("Negative reviews")
    print("Top R")
    print("Lead comm")
    print("Random comm <game> <n>")
    print("Game list")
    print("Compare <game1> | <game2>")
    print("0 - выход")


COMMANDS = {
    "positive reviews":
        lambda df: show_reviews(df, "Positive"),

    "negative reviews":
        lambda df: show_reviews(df, "Negative"),

    "top r":
        top_r_games,

    "lead comm":
        lead_comm,

    "game list":
        game_list
}


def handle_command(command: str, df: pd.DataFrame) -> bool:
    """
    Обрабатывает команды пользователя.

    Простые команды вызываются через словарь COMMANDS.
    Команды с параметрами обрабатываются отдельно.

    Returns
    -------
    bool
        True — продолжить работу программы.
        False — завершить выполнение.
    """

    command_lower = command.lower()

    try:

        if command_lower == "0":
            print("Работа завершена.")
            return False

        if command_lower in COMMANDS:
            COMMANDS[command_lower](df)
            return True

        if command_lower.startswith("random comm "):

            raw = command[len("Random comm "):]
            last_space = raw.rfind(" ")

            if last_space == -1:
                raise ValueError(
                    "Используйте: "
                    "Random comm <game> <n>"
                )

            game = raw[:last_space]
            count = int(raw[last_space + 1:])

            random_comm(df, game, count)
            return True

        if command_lower.startswith("compare "):

            raw = command[len("Compare "):]

            if "|" not in raw:
                raise ValueError(
                    "Используйте: "
                    "Compare game1 | game2"
                )

            game_1, game_2 = [
                item.strip()
                for item in raw.split("|", 1)
            ]

            compare_games(df, game_1, game_2)
            return True

        print("Неизвестная команда.")

    except ValueError as error:
        print(f"Ошибка: {error}")

    except KeyError:
        print("Ошибка структуры данных.")

    except Exception as error:
        print(f"Ошибка: {error}")

    return True


def main() -> None:
    """
    Главная функция программы.

    Загружает датасет, подготавливает отзывы
    и запускает цикл обработки команд.
    """

    try:

        df = load_dataset(DATASET_PATH)
        prepared_df = prepare_reviews(df)

        print(
            f"\nЗагружено отзывов: "
            f"{len(prepared_df)}"
        )

        while True:
            print_menu()

            command = input(
                "\nВведите команду: "
            )

            if not handle_command(
                command,
                prepared_df
            ):
                break

    except ValueError as error:
        print(f"Ошибка: {error}")

    except Exception as error:
        print(
            f"Критическая ошибка: "
            f"{error}"
        )


if __name__ == "__main__":
    main()


Загружено отзывов: 49999

Команды:
Positive reviews
Negative reviews
Top R
Lead comm
Random comm <game> <n>
Game list
Compare <game1> | <game2>
0 - выход

Введите команду: 0
Работа завершена.
